[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/085.%20The%20Uncapacitated%20Facility%20Location%20Problem/P85-Tier-4_executed.ipynb)

---


# 85. The Uncapacitated Facility Location Problem
## Tier 4: Online Learning with Multi-Armed Bandits

This notebook studies how a fixed set of facility-opening configurations can be selected over time when transportation costs vary. Each configuration is an option, or **arm**, in a multi-armed bandit problem. The Upper Confidence Bound (UCB) policy uses observed rewards to balance trying less-tested configurations with choosing configurations that have performed well.

### Objectives, inputs, and outputs

- **Objective:** learn a high-reward facility configuration through repeated decisions in a changing environment.
- **Inputs:** facility fixed costs, a base customer-to-facility transportation-cost matrix, and parameters describing environmental variation and UCB learning.
- **Outputs:** the selected configurations, observed rewards, a static-reference benchmark, cumulative benchmark gap, and plots that summarize learning behavior.
- **Solution sequence:** define the instance and environment; generate feasible configurations; identify a static reference configuration; run UCB; then analyze the run and selected parameter sensitivities.

### Background: the facility-location decision

A facility configuration states which facilities are open. For a given configuration, each customer is assigned to the cheapest open facility. The fixed cost of opening facilities and the resulting transportation costs together determine the configuration's total cost. This notebook uses the negative of that cost as a reward, so a larger reward means a lower cost.

The environment changes over time. A decision-maker observes the reward of the configuration chosen at each time step, but does not receive the rewards that the other configurations would have produced at that step. This partial feedback is the central feature of the bandit setting.

### Modeling notes and scope

- Facility capacities, facility-opening constraints beyond nonempty configurations, customer demand quantities, and customer-level assignment restrictions are not modeled.
- Customer demand variation is represented here by a time-varying multiplier on transportation costs; the model does not explicitly multiply costs by demand quantities.
- Cost changes are shared across all configurations at a given time step. Random shocks are generated once per time step and reused for all evaluations at that time, so comparisons refer to the same environmental state.
- Every nonempty subset of the six facilities is available as an arm. The assignment for each customer is recomputed using the current transportation costs when an arm is evaluated.
- The static reference arm is selected by evaluating all arms over the learning horizon. It is an analysis benchmark, not information supplied to the UCB policy while it makes decisions.
- The reward noise affects what UCB observes. Benchmark gaps use the underlying, noise-free costs so that observation noise is not mistaken for a change in operating cost.

### The UCB learning idea

An arm with a high observed average reward is attractive, while an arm with few observations has greater uncertainty and may deserve exploration. UCB combines these two quantities:

$$
UCB_a(t) = \bar{r}_a(t) + c\sqrt{\frac{\ln(t+1)}{n_a(t)}}.
$$

Here, $a$ identifies a facility configuration, $\bar{r}_a(t)$ is its observed average reward, $n_a(t)$ is the number of times it has been selected, and $c$ is the exploration constant. An unselected arm is assigned infinite priority so that it is tried. The policy selects an arm with the largest UCB value, observes its reward, and updates its statistics.

For this dynamic, noisy setting, the confidence term is used as a practical exploration heuristic. The classical UCB regret guarantees rely on assumptions such as stationary reward distributions and suitable reward bounds; those guarantees should not be inferred for this changing environment.

### What to examine

- How the cumulative gap to the static reference changes over time.
- Whether selected arms concentrate on configurations with high observed average reward.
- How many distinct configurations are selected during the horizon.
- How the exploration constant affects the same UCB policy, and how the environment scenarios affect its observed performance.

These plots and summaries describe the simulated run; they do not by themselves establish a general performance guarantee.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import itertools
import random

import matplotlib.pyplot as plt
import numpy as np

# A fixed seed makes the example reproducible when run from the first cell.
np.random.seed(85)
random.seed(85)

print("Libraries imported for online learning with multi-armed bandits.")

In [ ]:
@dataclass
class UFLPInstance:
    """Data for an uncapacitated facility location instance."""

    n_facilities: int
    n_customers: int
    fixed_costs: List[float]
    base_transport_costs: np.ndarray

    def __post_init__(self):
        """Check that the cost data match the declared instance dimensions."""
        if len(self.fixed_costs) != self.n_facilities:
            raise ValueError("There must be one fixed cost per facility.")
        if self.base_transport_costs.shape != (
            self.n_customers,
            self.n_facilities,
        ):
            raise ValueError(
                "The transportation-cost matrix must have shape "
                "(n_customers, n_facilities)."
            )


@dataclass
class UFLPSolution:
    """A facility configuration, customer assignment, and cost breakdown."""

    facilities_open: List[int]
    assignments: List[int]
    total_cost: float
    fixed_cost: float
    transport_cost: float
    feasible: bool


@dataclass
class BanditArm:
    """One candidate facility configuration and its learning statistics."""

    arm_id: int
    facilities_open: List[int]
    n_pulls: int = 0
    total_reward: float = 0.0
    avg_reward: float = 0.0
    confidence_bound: float = 0.0


@dataclass
class UCBParameters:
    """Parameters controlling the UCB learning run."""

    exploration_constant: float = 2.0
    time_steps: int = 500
    reward_noise_std: float = 5.0


@dataclass
class DynamicEnvironment:
    """Parameters controlling time variation in transportation costs."""

    demand_drift_amplitude: float = 0.1
    demand_drift_frequency: float = 100
    cost_drift_amplitude: float = 0.05
    cost_shock_probability: float = 0.02


print("Data structures defined for the UFLP and bandit model.")

In [ ]:
def create_online_learning_instance() -> Tuple[UFLPInstance, DynamicEnvironment]:
    """Create the six-facility, ten-customer example instance."""
    n_facilities = 6
    n_customers = 10
    fixed_costs = [80, 120, 95, 110, 85, 130]

    # This seed defines the example data independently of later simulation draws.
    np.random.seed(44)
    base_transport_costs = np.random.randint(
        15, 45, size=(n_customers, n_facilities)
    )

    # Each customer receives two lower-cost facility options in the base matrix.
    for customer in range(n_customers):
        preferred = np.random.choice(n_facilities, size=2, replace=False)
        for facility in preferred:
            base_transport_costs[customer, facility] = max(
                10, base_transport_costs[customer, facility] - 15
            )

    instance = UFLPInstance(
        n_facilities=n_facilities,
        n_customers=n_customers,
        fixed_costs=fixed_costs,
        base_transport_costs=base_transport_costs,
    )

    env_params = DynamicEnvironment(
        demand_drift_amplitude=0.1,
        demand_drift_frequency=100,
        cost_drift_amplitude=0.05,
        cost_shock_probability=0.02,
    )
    return instance, env_params


online_instance, dynamic_env_params = create_online_learning_instance()

print("Created the online-learning UFLP instance:")
print(f"  Facilities: {online_instance.n_facilities}")
print(f"  Customers: {online_instance.n_customers}")
print(f"  Fixed costs: {online_instance.fixed_costs}")
print(
    "  Base transportation-cost matrix shape: "
    f"{online_instance.base_transport_costs.shape}"
)
print("\nDynamic environment parameters:")
print(f"  Demand-drift amplitude: {dynamic_env_params.demand_drift_amplitude}")
print(f"  Demand-drift frequency: {dynamic_env_params.demand_drift_frequency}")
print(f"  Cost-drift amplitude: {dynamic_env_params.cost_drift_amplitude}")
print(f"  Cost-shock probability: {dynamic_env_params.cost_shock_probability}")

print("\nBase transportation-cost matrix:")
header = ["Customer"] + [f"F{j + 1}" for j in range(online_instance.n_facilities)]
print("\t".join(header))
for i, row in enumerate(online_instance.base_transport_costs):
    values = [f"C{i + 1}"] + [str(int(value)) for value in row]
    print("\t".join(values))

In [ ]:
class DynamicUFLPEnvironment:
    """Time-varying transportation costs for repeated UFLP decisions."""

    def __init__(self, instance: UFLPInstance, env_params: DynamicEnvironment):
        self.instance = instance
        self.env_params = env_params
        self.current_time = 0
        self._cost_cache: Dict[int, np.ndarray] = {}
        self.current_costs = self._generate_costs(0)
        self._cost_cache[0] = self.current_costs.copy()

    def _generate_costs(self, time_step: int) -> np.ndarray:
        """Build one shared cost matrix for a particular environmental time."""
        demand_multiplier = 1.0 + self.env_params.demand_drift_amplitude * np.sin(
            2 * np.pi * time_step / self.env_params.demand_drift_frequency
        )
        cost_multiplier = 1.0 + self.env_params.cost_drift_amplitude * np.sin(
            2
            * np.pi
            * time_step
            / (self.env_params.demand_drift_frequency * 2)
        )
        current_costs = (
            self.instance.base_transport_costs
            * demand_multiplier
            * cost_multiplier
        )

        # A shock is drawn once for this time step so every arm is evaluated
        # against the same realized environment state.
        if np.random.random() < self.env_params.cost_shock_probability:
            n_shocked = max(1, self.instance.n_facilities // 3)
            shocked_facilities = np.random.choice(
                self.instance.n_facilities,
                size=n_shocked,
                replace=False,
            )
            for facility in shocked_facilities:
                shock_factor = np.random.uniform(0.8, 1.3)
                current_costs[:, facility] *= shock_factor

        return current_costs

    def set_time(self, time_step: int) -> None:
        """Set the environment time while reusing its realized cost matrix."""
        if time_step < 0:
            raise ValueError("time_step must be nonnegative.")
        self.current_time = time_step
        if time_step not in self._cost_cache:
            self._cost_cache[time_step] = self._generate_costs(time_step)
        self.current_costs = self._cost_cache[time_step].copy()

    def get_current_costs(self) -> np.ndarray:
        """Return a copy of the transportation costs at the current time."""
        return self.current_costs.copy()

    def calculate_solution_cost(
        self,
        facilities_open: List[int],
        assignments: List[int],
    ) -> float:
        """Calculate fixed plus transportation cost in the current state."""
        if len(facilities_open) != self.instance.n_facilities:
            raise ValueError("There must be one opening decision per facility.")
        if not any(facilities_open):
            raise ValueError("At least one facility must be open.")
        if len(assignments) != self.instance.n_customers:
            raise ValueError("There must be one assignment per customer.")

        fixed_cost = sum(
            self.instance.fixed_costs[j]
            for j in range(self.instance.n_facilities)
            if facilities_open[j]
        )
        for facility in assignments:
            if facility < 0 or facility >= self.instance.n_facilities:
                raise ValueError("An assignment refers to an invalid facility.")
            if not facilities_open[facility]:
                raise ValueError("Every customer must be assigned to an open facility.")

        transport_cost = sum(
            self.current_costs[customer, assignments[customer]]
            for customer in range(self.instance.n_customers)
        )
        return float(fixed_cost + transport_cost)

    def step(self) -> None:
        """Advance the environment by one time step."""
        self.set_time(self.current_time + 1)


print("Dynamic UFLP environment defined.")

In [ ]:
class FacilityConfigurationGenerator:
    """Generate nonempty facility configurations and their customer assignments."""

    def __init__(self, instance: UFLPInstance):
        self.instance = instance

    def assign_customers_to_facilities(
        self,
        facilities_open: List[int],
        transport_costs: np.ndarray,
    ) -> List[int]:
        """Assign each customer to its cheapest open facility."""
        if len(facilities_open) != self.instance.n_facilities:
            raise ValueError("There must be one opening decision per facility.")
        if not any(facilities_open):
            raise ValueError("At least one facility must be open.")
        if transport_costs.shape != (
            self.instance.n_customers,
            self.instance.n_facilities,
        ):
            raise ValueError("Transportation costs have an incompatible shape.")

        assignments = []
        for customer in range(self.instance.n_customers):
            open_facilities = [
                facility
                for facility in range(self.instance.n_facilities)
                if facilities_open[facility]
            ]
            best_facility = min(
                open_facilities,
                key=lambda facility: transport_costs[customer, facility],
            )
            assignments.append(best_facility)
        return assignments

    def is_feasible_configuration(self, facilities_open: List[int]) -> bool:
        """A configuration is feasible when at least one facility is open."""
        return (
            len(facilities_open) == self.instance.n_facilities
            and any(facilities_open)
            and all(value in (0, 1) for value in facilities_open)
        )

    def generate_feasible_configurations(
        self,
        max_configurations: int = 100,
    ) -> List[List[int]]:
        """Generate small configurations first, then sample larger ones."""
        if max_configurations <= 0:
            return []

        configurations = []
        # Small configurations are listed first to preserve the intended
        # emphasis on opening relatively few facilities.
        for n_open in range(1, min(4, self.instance.n_facilities + 1)):
            for open_set in itertools.combinations(
                range(self.instance.n_facilities), n_open
            ):
                config = [0] * self.instance.n_facilities
                for facility in open_set:
                    config[facility] = 1
                configurations.append(config)
                if len(configurations) >= max_configurations:
                    return configurations

        # There are finitely many nonempty configurations. Sampling without
        # replacement avoids an endless search when the requested limit is
        # larger than the number of feasible configurations.
        remaining = []
        for n_open in range(4, self.instance.n_facilities + 1):
            for open_set in itertools.combinations(
                range(self.instance.n_facilities), n_open
            ):
                config = [0] * self.instance.n_facilities
                for facility in open_set:
                    config[facility] = 1
                remaining.append(config)

        np.random.shuffle(remaining)
        n_needed = max_configurations - len(configurations)
        configurations.extend(remaining[:n_needed])
        return configurations

    def create_bandit_arms(self, max_arms: int = 100) -> List[BanditArm]:
        """Represent each generated feasible configuration as one bandit arm."""
        configurations = self.generate_feasible_configurations(max_arms)
        return [
            BanditArm(arm_id=arm_id, facilities_open=config)
            for arm_id, config in enumerate(configurations)
        ]


print("Facility-configuration generator defined.")

## Mathematical model and learning state

Let $I=\{1,\ldots,n\}$ be the customers and $J=\{1,\ldots,m\}$ the candidate facilities. For a time step $t$, let $f_j$ be the fixed cost of opening facility $j$, and let $c_{ij}(t)$ be the transportation cost from customer $i$ to facility $j$. The binary decision $y_j$ indicates whether facility $j$ is open, and $x_{ij}$ indicates whether customer $i$ is assigned to facility $j$.

For a fixed environmental state, the uncapacitated facility-location cost is

$$
\begin{aligned}
\min \quad & \sum_{j\in J} f_j y_j + \sum_{i\in I}\sum_{j\in J} c_{ij}(t)x_{ij} \\
\text{subject to} \quad
 & \sum_{j\in J} x_{ij}=1 && \text{for each } i\in I, \\
 & x_{ij}\leq y_j && \text{for each } i\in I,\ j\in J, \\
 & x_{ij}\in\{0,1\},\quad y_j\in\{0,1\}, \\
 & \sum_{j\in J}y_j\geq 1.
\end{aligned}
$$

The first constraint assigns each customer exactly once; the second allows assignment only to an open facility; the final constraint rules out the empty configuration. Given an open set, the least-cost assignment is found independently for each customer by choosing its cheapest open facility. This is the operation implemented by `assign_customers_to_facilities`.

The candidate arm set contains every nonempty facility subset. Its size is $K=2^m-1$. For arm $a$, let $C_a(t)$ be the cost after making the cheapest assignments at time $t$. The reward observed by the learner is

$$
R_a(t)=-C_a(t)+\varepsilon_t,
$$

where $\varepsilon_t$ is zero-mean Gaussian observation noise with the configured standard deviation. The noise is part of the learner's observation, not part of the underlying operating cost.

For each arm, the state consists of its pull count $n_a$ and cumulative reward $S_a$. Its empirical mean is $\bar r_a=S_a/n_a$ when $n_a>0$. At a decision time, the UCB score is

$$
UCB_a(t)=
\begin{cases}
+\infty, & n_a=0,\\
\bar r_a+c\sqrt{\dfrac{\ln(t+1)}{n_a}}, & n_a>0.
\end{cases}
$$

The policy chooses an arm with the largest score. After observing its reward, it updates $n_a\leftarrow n_a+1$ and $S_a\leftarrow S_a+R_a(t)$. The infinite score ensures every arm is initially sampled. The code stores these quantities in `BanditArm.n_pulls`, `BanditArm.total_reward`, and `BanditArm.avg_reward`; `calculate_ucb_value`, `select_arm`, and `update_arm_statistics` implement the score, choice, and update.

Generating all arms takes $O(2^m)$ configurations. Evaluating one configuration's assignments takes $O(nm)$ in the direct implementation, and scanning all $K$ arms for a UCB choice takes $O(K)$. Thus the approach is practical for this small example but the number of configurations grows exponentially with the number of facilities.

In [ ]:
class UCBBanditAlgorithm:
    """Apply UCB to a finite set of facility configurations."""

    def __init__(
        self,
        instance: UFLPInstance,
        env: DynamicUFLPEnvironment,
        ucb_params: UCBParameters,
    ):
        if ucb_params.time_steps <= 0:
            raise ValueError("time_steps must be positive.")
        self.instance = instance
        self.env = env
        self.params = ucb_params
        self.config_generator = FacilityConfigurationGenerator(instance)
        self.arms = self.config_generator.create_bandit_arms(max_arms=100)
        if not self.arms:
            raise ValueError("At least one feasible arm is required.")

        self.current_time = 0
        self.total_reward = 0.0
        self.cumulative_regret = []
        self.arm_selections = []
        self.rewards_history = []
        self.optimal_reward_history = []
        self.optimal_arm_id = self.find_optimal_arm()

        print(f"Generated {len(self.arms)} feasible facility configurations.")
        print(f"Static reference arm ID: {self.optimal_arm_id}")

    def _noise_free_arm_reward(self, arm: BanditArm) -> float:
        """Return the negative operating cost in the current state."""
        costs = self.env.get_current_costs()
        assignments = self.config_generator.assign_customers_to_facilities(
            arm.facilities_open,
            costs,
        )
        total_cost = self.env.calculate_solution_cost(
            arm.facilities_open,
            assignments,
        )
        return -total_cost

    def find_optimal_arm(self) -> int:
        """Find the best fixed arm over the learning horizon for benchmarking."""
        total_rewards = np.zeros(len(self.arms), dtype=float)
        for time_step in range(self.params.time_steps):
            self.env.set_time(time_step)
            for arm_index, arm in enumerate(self.arms):
                total_rewards[arm_index] += self._noise_free_arm_reward(arm)

        # This full-horizon comparator is used only to assess the run; the
        # selection policy receives no information about its identity.
        best_arm_id = int(np.argmax(total_rewards))
        self.env.set_time(0)
        return best_arm_id

    def calculate_ucb_value(self, arm: BanditArm) -> float:
        """Compute the empirical reward plus the UCB exploration bonus."""
        if arm.n_pulls == 0:
            return float("inf")

        confidence_term = self.params.exploration_constant * np.sqrt(
            np.log(self.current_time + 1) / arm.n_pulls
        )
        arm.confidence_bound = float(confidence_term)
        return arm.avg_reward + confidence_term

    def select_arm(self) -> BanditArm:
        """Select an arm with the largest current UCB score."""
        return max(self.arms, key=self.calculate_ucb_value)

    def get_arm_reward(self, arm: BanditArm) -> float:
        """Return a noisy observation of the selected arm's reward."""
        noise = np.random.normal(0, self.params.reward_noise_std)
        return self._noise_free_arm_reward(arm) + noise

    def update_arm_statistics(self, arm: BanditArm, reward: float) -> None:
        """Update the selected arm's sample count and empirical reward."""
        arm.n_pulls += 1
        arm.total_reward += reward
        arm.avg_reward = arm.total_reward / arm.n_pulls

    def calculate_optimal_reward(self) -> float:
        """Return the noise-free reward of the static reference arm now."""
        reference_arm = self.arms[self.optimal_arm_id]
        return self._noise_free_arm_reward(reference_arm)

    def run_learning(self) -> Dict:
        """Run UCB sequentially and record rewards and benchmark gaps."""
        print("Starting online learning with UCB.")
        print(f"Time steps: {self.params.time_steps}")
        print(f"Exploration constant: {self.params.exploration_constant}")

        for time_step in range(self.params.time_steps):
            self.current_time = time_step
            self.env.set_time(time_step)

            selected_arm = self.select_arm()
            observed_reward = self.get_arm_reward(selected_arm)
            true_reward = self._noise_free_arm_reward(selected_arm)
            reference_reward = self.calculate_optimal_reward()

            self.update_arm_statistics(selected_arm, observed_reward)
            self.total_reward += observed_reward
            self.arm_selections.append(selected_arm.arm_id)
            self.rewards_history.append(observed_reward)
            self.optimal_reward_history.append(reference_reward)

            # The benchmark gap excludes observation noise. It can be
            # negative at individual times, while its full-horizon sum is
            # nonnegative up to floating-point error because the reference
            # is the best fixed arm over this same horizon.
            step_gap = reference_reward - true_reward
            previous_gap = self.cumulative_regret[-1] if self.cumulative_regret else 0.0
            self.cumulative_regret.append(previous_gap + step_gap)

            if (time_step + 1) % 100 == 0:
                average_reward = self.total_reward / (time_step + 1)
                print(
                    f"Time {time_step + 1}: average observed reward = "
                    f"{average_reward:.2f}, cumulative benchmark gap = "
                    f"{self.cumulative_regret[-1]:.2f}"
                )

        best_observed_arm = max(self.arms, key=lambda arm: arm.avg_reward)
        results = {
            "final_avg_reward": self.total_reward / self.params.time_steps,
            "final_cumulative_regret": self.cumulative_regret[-1],
            "arms_explored": len(set(self.arm_selections)),
            "exploration_rate": len(set(self.arm_selections)) / len(self.arms),
            "best_arm": best_observed_arm,
            "optimal_arm": self.arms[self.optimal_arm_id],
        }
        return results


ucb_params = UCBParameters(
    exploration_constant=2.0,
    time_steps=500,
    reward_noise_std=5.0,
)
dynamic_env = DynamicUFLPEnvironment(online_instance, dynamic_env_params)
ucb_bandit = UCBBanditAlgorithm(online_instance, dynamic_env, ucb_params)
learning_results = ucb_bandit.run_learning()

print("\n=== ONLINE LEARNING RESULTS ===")
print(f"Generated arms: {len(ucb_bandit.arms)}")
print(f"Final average observed reward: {learning_results['final_avg_reward']:.2f}")
print(
    "Final cumulative gap to the static reference: "
    f"{learning_results['final_cumulative_regret']:.2f}"
)
print(
    f"Distinct arms selected: {learning_results['arms_explored']}/"
    f"{len(ucb_bandit.arms)}"
)
print(f"Fraction of arms selected: {learning_results['exploration_rate']:.2%}")

for label, arm in (
    ("Best observed arm", learning_results["best_arm"]),
    ("Static reference arm", learning_results["optimal_arm"]),
):
    open_facilities = [
        facility + 1
        for facility, is_open in enumerate(arm.facilities_open)
        if is_open
    ]
    print(f"\n{label}:")
    print(f"  Arm ID: {arm.arm_id}")
    print(f"  Open facilities: {open_facilities}")
    print(f"  Average observed reward: {arm.avg_reward:.2f}")
    print(f"  Number of selections: {arm.n_pulls}")

assert len(ucb_bandit.arm_selections) == ucb_params.time_steps
assert len(ucb_bandit.cumulative_regret) == ucb_params.time_steps
assert sum(arm.n_pulls for arm in ucb_bandit.arms) == ucb_params.time_steps

## Interpreting the run

The learner's average reward is based on noisy observations, so it need not equal the negative average operating cost. The arm labeled **Best observed arm** is the arm with the largest sample-average observed reward; it is not guaranteed to be the best arm in every environmental state.

The static reference is the single configuration with the greatest total noise-free reward over the full simulated horizon. The plotted cumulative benchmark gap adds the difference between that reference's noise-free reward and the selected arm's noise-free reward at each time. Individual increments can be negative because a changing environment can temporarily favor another configuration. Across the full horizon, the final sum should be nonnegative apart from floating-point effects, since the reference is the best fixed arm over that same horizon.

The number of distinct selected arms measures how many configurations the policy tried at least once. It is not the same as the number of decisions devoted to exploration: a configuration may be selected repeatedly after its initial trial.

In [ ]:
def visualize_online_learning(
    ucb_bandit: UCBBanditAlgorithm,
    results: Dict,
) -> None:
    """Plot the recorded UCB learning history and summarize its behavior."""
    if not ucb_bandit.cumulative_regret:
        raise ValueError("Run the learning process before plotting its history.")

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Online Learning with UCB for Facility Location",
        fontsize=16,
        fontweight="bold",
    )
    time_steps = np.arange(1, len(ucb_bandit.cumulative_regret) + 1)

    ax1 = axes[0, 0]
    ax1.plot(
        time_steps,
        ucb_bandit.cumulative_regret,
        linewidth=2,
        color="#2E86AB",
        label="Cumulative benchmark gap",
    )
    ax1.set_xlabel("Time step")
    ax1.set_ylabel("Cumulative noise-free reward gap")
    ax1.set_title("Cumulative Gap to the Static Reference")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2 = axes[0, 1]
    selected_costs = [-reward for reward in ucb_bandit.rewards_history]
    reference_costs = [-reward for reward in ucb_bandit.optimal_reward_history]
    ax2.plot(
        time_steps,
        selected_costs,
        linewidth=1,
        color="#F18F01",
        alpha=0.7,
        label="Selected arm (includes observation noise)",
    )
    ax2.plot(
        time_steps,
        reference_costs,
        linewidth=2,
        color="#2E86AB",
        label="Static reference (noise-free)",
    )
    ax2.set_xlabel("Time step")
    ax2.set_ylabel("Cost-equivalent value")
    ax2.set_title("Observed and Reference Cost Histories")
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    ax3 = axes[1, 0]
    sample_period = max(1, len(ucb_bandit.arm_selections) // 100)
    sampled_indices = np.arange(0, len(ucb_bandit.arm_selections), sample_period)
    sampled_selections = np.asarray(ucb_bandit.arm_selections)[sampled_indices]
    ax3.scatter(
        sampled_indices + 1,
        sampled_selections,
        alpha=0.65,
        s=20,
        color="#A23B72",
        label="Selected arm",
    )
    ax3.axhline(
        results["best_arm"].arm_id,
        color="#F18F01",
        linestyle="--",
        label=f"Best observed arm ({results['best_arm'].arm_id})",
    )
    ax3.axhline(
        results["optimal_arm"].arm_id,
        color="#2E86AB",
        linestyle="--",
        label=f"Static reference arm ({results['optimal_arm'].arm_id})",
    )
    ax3.set_xlabel("Time step")
    ax3.set_ylabel("Arm ID")
    ax3.set_title("Selected Arms (Sampled for Readability)")
    ax3.grid(True, alpha=0.3)
    ax3.legend()

    ax4 = axes[1, 1]
    arm_pulls = [arm.n_pulls for arm in ucb_bandit.arms]
    arm_ids = [arm.arm_id for arm in ucb_bandit.arms]
    average_rewards = np.asarray([arm.avg_reward for arm in ucb_bandit.arms])
    mean_reward = float(np.mean(average_rewards))
    colors = [
        "#2E86AB" if reward >= mean_reward else "#A23B72"
        for reward in average_rewards
    ]
    ax4.bar(arm_ids, arm_pulls, color=colors, alpha=0.75)
    ax4.axvline(
        results["best_arm"].arm_id,
        color="#F18F01",
        linestyle="--",
        label="Best observed arm",
    )
    ax4.axvline(
        results["optimal_arm"].arm_id,
        color="#2E86AB",
        linestyle="--",
        label="Static reference arm",
    )
    ax4.set_xlabel("Arm ID")
    ax4.set_ylabel("Number of selections")
    ax4.set_title("Selections by Arm")
    ax4.grid(True, alpha=0.3)
    ax4.legend()

    reward_legend = [
        plt.Line2D([0], [0], color="#2E86AB", lw=6, label="At or above mean observed reward"),
        plt.Line2D([0], [0], color="#A23B72", lw=6, label="Below mean observed reward"),
    ]
    ax4.legend(handles=ax4.get_legend_handles_labels()[0] + reward_legend)

    plt.tight_layout()
    plt.show()

    final_gap = ucb_bandit.cumulative_regret[-1]
    average_gap_per_step = final_gap / len(ucb_bandit.cumulative_regret)
    unique_arms = len(set(ucb_bandit.arm_selections))
    print("\n=== LEARNING SUMMARY ===")
    print(f"Final cumulative noise-free benchmark gap: {final_gap:.2f}")
    print(f"Average cumulative-gap increment per step: {average_gap_per_step:.4f}")
    print(f"Distinct arms selected: {unique_arms}/{len(ucb_bandit.arms)}")
    print(
        "The plots summarize this simulated run; observed patterns depend on "
        "the environment and random reward noise."
    )


visualize_online_learning(ucb_bandit, learning_results)

## Sensitivity to the exploration constant

The exploration constant $c$ controls the weight of the uncertainty bonus in the UCB score. A larger value gives less-tested arms more priority; a smaller value places relatively more weight on observed average rewards. The following experiment keeps the UCB algorithm and learning horizon fixed while varying only $c$.

The runs use the same random seed so that the generated arm order, environmental shocks, and reward-noise stream are reproducible across the tested values. The value used in the main example is included for reference. Differences in this finite simulation illustrate parameter sensitivity; they do not establish that one constant is best in general.

In [ ]:
def analyze_ucb_exploration_sensitivity(
    instance: UFLPInstance,
    env_params: DynamicEnvironment,
    exploration_constants: List[float],
    time_steps: int,
    reward_noise_std: float,
) -> Dict[float, Dict]:
    """Run the same UCB method at several exploration-constant values."""
    if not exploration_constants:
        raise ValueError("At least one exploration constant is required.")

    saved_numpy_state = np.random.get_state()
    sensitivity_results = {}
    try:
        for exploration_constant in exploration_constants:
            np.random.seed(85)
            env = DynamicUFLPEnvironment(instance, env_params)
            params = UCBParameters(
                exploration_constant=exploration_constant,
                time_steps=time_steps,
                reward_noise_std=reward_noise_std,
            )
            bandit = UCBBanditAlgorithm(instance, env, params)
            results = bandit.run_learning()
            sensitivity_results[exploration_constant] = {
                "final_gap": results["final_cumulative_regret"],
                "exploration_rate": results["exploration_rate"],
                "regret_history": bandit.cumulative_regret.copy(),
            }
    finally:
        np.random.set_state(saved_numpy_state)

    return sensitivity_results


sensitivity_results = analyze_ucb_exploration_sensitivity(
    instance=online_instance,
    env_params=dynamic_env_params,
    exploration_constants=[1.0, 2.0, 3.0],
    time_steps=ucb_params.time_steps,
    reward_noise_std=ucb_params.reward_noise_std,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for exploration_constant, result in sensitivity_results.items():
    steps = np.arange(1, len(result["regret_history"]) + 1)
    axes[0].plot(
        steps,
        result["regret_history"],
        linewidth=2,
        label=f"c = {exploration_constant:g}",
    )
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("Cumulative noise-free benchmark gap")
axes[0].set_title("UCB Benchmark Gap by Exploration Constant")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

constants = list(sensitivity_results)
final_gaps = [sensitivity_results[value]["final_gap"] for value in constants]
exploration_rates = [
    100 * sensitivity_results[value]["exploration_rate"]
    for value in constants
]
x = np.arange(len(constants))
width = 0.36
axes[1].bar(x - width / 2, final_gaps, width, label="Final cumulative gap")
axes[1].set_xlabel("Exploration constant, c")
axes[1].set_ylabel("Final cumulative noise-free benchmark gap")
axes[1].set_title("Final Gap and Distinct-Arm Selection Rate")
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"{value:g}" for value in constants])
axes[1].grid(True, axis="y", alpha=0.3)
rate_axis = axes[1].twinx()
rate_axis.bar(
    x + width / 2,
    exploration_rates,
    width,
    color="#A23B72",
    alpha=0.75,
    label="Distinct arms selected (%)",
)
rate_axis.set_ylabel("Distinct arms selected (%)")
handles_left, labels_left = axes[1].get_legend_handles_labels()
handles_right, labels_right = rate_axis.get_legend_handles_labels()
axes[1].legend(
    handles_left + handles_right,
    labels_left + labels_right,
    loc="best",
)
plt.tight_layout()
plt.show()

print("\n=== UCB EXPLORATION-CONSTANT SENSITIVITY ===")
for exploration_constant, result in sensitivity_results.items():
    print(
        f"c = {exploration_constant:g}: "
        f"final cumulative gap = {result['final_gap']:.2f}, "
        f"distinct-arm selection rate = {result['exploration_rate']:.2%}"
    )

## Sensitivity to environmental dynamics

This experiment applies the same UCB policy to four specified environments. The scenarios differ in the amplitudes and frequency of cost variation and in the probability of a cost shock. Each run uses the same number of learning steps and the same UCB settings.

The displayed exploration rate is the fraction of available arms selected at least once. It is a descriptive measure of arm coverage, not a direct measurement of how much time the policy spent exploring. Interpret differences in final benchmark gaps in light of each scenario's realized cost path; no ordering of scenario performance is guaranteed in advance.

In [ ]:
def analyze_dynamic_adaptation(
    instance: UFLPInstance,
    base_env_params: DynamicEnvironment,
) -> Dict[str, Dict]:
    """Measure UCB behavior under several specified environment scenarios."""
    scenarios = {
        "Stable": DynamicEnvironment(0.01, 1000, 0.01, 0.01),
        "Moderate": DynamicEnvironment(0.1, 100, 0.05, 0.02),
        "Volatile": DynamicEnvironment(0.2, 50, 0.1, 0.05),
        "Chaotic": DynamicEnvironment(0.3, 25, 0.15, 0.1),
    }
    scenario_results = {}

    for scenario_name, scenario_params in scenarios.items():
        print(f"\nRunning the {scenario_name.lower()} environment scenario.")
        env = DynamicUFLPEnvironment(instance, scenario_params)
        params = UCBParameters(
            exploration_constant=ucb_params.exploration_constant,
            time_steps=300,
            reward_noise_std=ucb_params.reward_noise_std,
        )
        bandit = UCBBanditAlgorithm(instance, env, params)
        results = bandit.run_learning()
        scenario_results[scenario_name] = {
            "final_gap": results["final_cumulative_regret"],
            "exploration_rate": results["exploration_rate"],
            "average_reward": results["final_avg_reward"],
            "gap_history": bandit.cumulative_regret.copy(),
        }
        print(f"  Final cumulative benchmark gap: {results['final_cumulative_regret']:.2f}")
        print(f"  Distinct-arm selection rate: {results['exploration_rate']:.2%}")
        print(f"  Average observed reward: {results['final_avg_reward']:.2f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for scenario_name, result in scenario_results.items():
        steps = np.arange(1, len(result["gap_history"]) + 1)
        axes[0].plot(
            steps,
            result["gap_history"],
            linewidth=2,
            label=scenario_name,
        )
    axes[0].set_xlabel("Time step")
    axes[0].set_ylabel("Cumulative noise-free benchmark gap")
    axes[0].set_title("Benchmark Gap by Environment Scenario")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    scenario_names = list(scenario_results)
    final_gaps = [scenario_results[name]["final_gap"] for name in scenario_names]
    exploration_rates = [
        100 * scenario_results[name]["exploration_rate"]
        for name in scenario_names
    ]
    x = np.arange(len(scenario_names))
    width = 0.36
    axes[1].bar(
        x - width / 2,
        final_gaps,
        width,
        color="#2E86AB",
        label="Final cumulative gap",
    )
    axes[1].set_xlabel("Environment scenario")
    axes[1].set_ylabel("Final cumulative noise-free benchmark gap")
    axes[1].set_title("Final Gap and Distinct-Arm Selection Rate")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(scenario_names)
    axes[1].grid(True, axis="y", alpha=0.3)
    rate_axis = axes[1].twinx()
    rate_axis.bar(
        x + width / 2,
        exploration_rates,
        width,
        color="#A23B72",
        alpha=0.75,
        label="Distinct arms selected (%)",
    )
    rate_axis.set_ylabel("Distinct arms selected (%)")
    left_handles, left_labels = axes[1].get_legend_handles_labels()
    right_handles, right_labels = rate_axis.get_legend_handles_labels()
    axes[1].legend(
        left_handles + right_handles,
        left_labels + right_labels,
        loc="best",
    )
    plt.tight_layout()
    plt.show()

    print("\n=== ENVIRONMENT-SCENARIO SUMMARY ===")
    print(f"{'Scenario':<12} {'Final gap':>14} {'Distinct arms':>18}")
    print("-" * 46)
    for scenario_name, result in scenario_results.items():
        print(
            f"{scenario_name:<12} "
            f"{result['final_gap']:>14.2f} "
            f"{result['exploration_rate']:>17.1%}"
        )
    print(
        "Scenario comparisons describe these simulated runs; changes in "
        "environmental variation do not imply a guaranteed change in UCB behavior."
    )
    return scenario_results


scenario_results = analyze_dynamic_adaptation(
    online_instance,
    dynamic_env_params,
)

## Strengths, limitations, and appropriate use

### Strengths

- UCB makes sequential choices using only the rewards observed from selected configurations.
- The exploration bonus gives less-tested configurations an explicit opportunity to be evaluated.
- The method can continue to make decisions as the modeled environment changes, without solving a new static optimization problem at every step.
- The finite arm set and recorded learning history make the decision process inspectable.

### Limitations

- The configuration set grows exponentially with the number of facilities, so enumerating every nonempty subset is suitable only for modest instances.
- UCB does not guarantee that each decision is optimal for the current environmental state. In a changing environment, evidence from earlier times may be less informative about current costs.
- The confidence term is a practical heuristic here; standard stationary-bandit guarantees do not automatically apply to this dynamic reward process.
- The static reference uses the complete simulated horizon and is an evaluation benchmark, not a deployable online decision rule.
- The cost model omits capacities, demand quantities, and other operational constraints. These omissions limit how directly the simulation can represent a real supply chain.

### When this modeling approach is useful

A bandit formulation is relevant when decisions are repeated, only the outcome of the chosen configuration is observed, and learning from those outcomes matters. In practice, the arm set, reward definition, environmental assumptions, and interpretation of the benchmark must match the operational problem. Natural extensions include adding problem-specific feasibility constraints or studying larger instances; those extensions require revisiting how candidate configurations are represented and evaluated.